
# MemoryBandit Multivariate Granger Causality (Memory Outcome)

This notebook runs multivariate state-space Granger causality analyses split by **memory outcome** (hits, misses, correct_rejections, false_alarms) from `combined_df`. Event can be toggled to `cue_on` (cue-aligned) or `feedback_start`; behavioral data always comes from `combined_df`. All reusable logic is in `mv_granger_utils.py`.



In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plot_config import set_plot_style
from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
    run_or_load_subject_gc_memory_outcome_df,
    load_gc_memory_outcome_df_cache,
    merge_with_behavior_memory_outcome,
    validate_smoke_outputs_memory_outcome,
    generate_subject_plot_memory_outcome,
    plot_smoke_memory_outcome,
    plot_aggregate_memory_outcome,
    fit_mixed_effects_memory_outcome,
    plot_mixed_effects_memory_outcome,
)

warnings.filterwarnings('ignore')
set_plot_style()


In [ ]:

# Paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_hitmiss'
os.makedirs(savepath, exist_ok=True)

# Subjects from demographics (same pattern as original notebook)
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().unique().tolist())

# Load + preprocess behavior
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print('Subjects:', len(subjects))
print('learn_df:', learn_df.shape)
print('combined_df:', combined_df.shape)
print('combined_df columns check:', {'participant','condition','trials_gamble','hits','memory_outcome'}.issubset(combined_df.columns))



In [ ]:
# Analysis configuration (use 'cue_on' for cue-aligned epochs; behavioral data always from combined_df)
ANALYSIS = {
    'event': 'cue_on',  # or 'cue_on'
    'day': 'Day1',
    'tmin': 0.0,
    'tmax': 1.0,
    'target_sfreq': 50,
    'fmin': 8,
    'fmax': 13,
    'gc_n_lags': 10,
    'n_freqs': None,
    'min_trials': 5,  # consider lowering for 4-category memory_outcome split if needed
}

# Surrogate configuration (source-shuffle null)
USE_SURROGATES = True
N_SURROGATES_SMOKE = 100
N_SURROGATES_BATCH = 100
SURROGATE_TAIL = 'greater'   # one-sided
SURROGATE_RANDOM_STATE = 42
N_JOBS_SURROGATES_SMOKE = 12
N_JOBS_SURROGATES_BATCH = 12
SURROGATE_PARALLEL_BACKEND = 'loky'

# DataFrame cache configuration (memory_outcome split)
GC_DF_CACHE_DIR = os.path.join(savepath, 'gc_df_cache_memory_outcome')
OVERWRITE_GC_DF_CACHE = False

ROI_PAIRS = [
    ('HPC', 'ACC'),
    ('HPC', 'AMY'),
    ('HPC', 'dlPFC'),
    ('HPC', 'dmPFC'),
    ('HPC', 'OFC'),
]

RUN_SINGLE_SUBJECT_SMOKE = True
SMOKE_SUBJECT = 'MS012' if 'MS012' in subjects else subjects[0]
SMOKE_ROI_SOURCE, SMOKE_ROI_TARGET = ROI_PAIRS[0]

print('Smoke subject:', SMOKE_SUBJECT)
print('Smoke pair:', SMOKE_ROI_SOURCE, '->', SMOKE_ROI_TARGET)
print('Surrogates enabled:', USE_SURROGATES)
print('Smoke surrogates:', N_SURROGATES_SMOKE, '| Batch surrogates:', N_SURROGATES_BATCH)
print('GC DF cache dir:', GC_DF_CACHE_DIR)
print('Overwrite cache:', OVERWRITE_GC_DF_CACHE)

ANALYSIS_RESULTS = globals().get('ANALYSIS_RESULTS', {})
SMOKE_RESULTS = globals().get('SMOKE_RESULTS', {})


def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')


def _normalize_roi_pairs(pairs):
    return sorted([list(pair) for pair in pairs])


def _build_analysis_tag(analysis, roi_pairs):
    roi_pairs_norm = _normalize_roi_pairs(roi_pairs)
    roi_hash = hashlib.sha1(json.dumps(roi_pairs_norm, sort_keys=True).encode('utf-8')).hexdigest()[:10]
    band_tag_local = f"{analysis['fmin']}-{analysis['fmax']}Hz"
    return (
        f"{analysis['event']}_{analysis['day']}"
        f"_t{_analysis_value_token(analysis['tmin'])}-{_analysis_value_token(analysis['tmax'])}"
        f"_sf{_analysis_value_token(analysis['target_sfreq'])}"
        f"_{band_tag_local}"
        f"_lag{_analysis_value_token(analysis['gc_n_lags'])}"
        f"_roi{roi_hash}"
    )


analysis_tag = _build_analysis_tag(ANALYSIS, ROI_PAIRS)
fmin_tag = f"{ANALYSIS['fmin']}Hz"
band_tag = f"{ANALYSIS['fmin']}-{ANALYSIS['fmax']}Hz"
ANALYSIS_DIR = os.path.join(savepath, 'notebook_runs', analysis_tag)
os.makedirs(ANALYSIS_DIR, exist_ok=True)


def get_analysis_paths(tag=None, source_roi=None):
    tag = analysis_tag if tag is None else tag
    base_dir = os.path.join(savepath, 'notebook_runs', tag)
    os.makedirs(base_dir, exist_ok=True)
    paths = {
        'base_dir': base_dir,
        'all_gc_csv': os.path.join(base_dir, 'all_pairs_mvgc_memory_outcome.csv'),
        'cache_log_csv': os.path.join(base_dir, 'cache_log_memory_outcome.csv'),
        'merged_gc_csv': os.path.join(base_dir, 'merged_gc_memory_outcome.csv'),
        'smoke_df_csv': os.path.join(base_dir, 'smoke_gc_memory_outcome.csv'),
        'smoke_meta_json': os.path.join(base_dir, 'smoke_gc_memory_outcome_meta.json'),
    }
    if source_roi is not None:
        paths['stats_csv'] = os.path.join(base_dir, f'mixed_model_gc_memory_outcome_stats_{source_roi}.csv')
    return paths


analysis_paths = get_analysis_paths()
print('Analysis tag:', analysis_tag)
print('Analysis dir:', ANALYSIS_DIR)
print('Filename band tag:', band_tag)


In [ ]:
# Smoke run + validation + smoke plot
smoke_pack = SMOKE_RESULTS.get(analysis_tag)
if smoke_pack is not None:
    print('Loaded smoke pack from in-memory registry for', analysis_tag)
elif RUN_SINGLE_SUBJECT_SMOKE:
    smoke_df, smoke_result, smoke_meta = run_or_load_subject_gc_memory_outcome_df(
        subj_id=SMOKE_SUBJECT,
        roi_source=SMOKE_ROI_SOURCE,
        roi_target=SMOKE_ROI_TARGET,
        base_dir=base_dir,
        combined_df=combined_df,
        cache_dir=GC_DF_CACHE_DIR,
        overwrite_cache=OVERWRITE_GC_DF_CACHE,
        return_result_dict=True,
        event=ANALYSIS['event'],
        day=ANALYSIS['day'],
        tmin=ANALYSIS['tmin'],
        tmax=ANALYSIS['tmax'],
        target_sfreq=ANALYSIS['target_sfreq'],
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
        gc_n_lags=ANALYSIS['gc_n_lags'],
        n_freqs=ANALYSIS['n_freqs'],
        min_trials=ANALYSIS['min_trials'],
        use_surrogates=USE_SURROGATES,
        n_surrogates=N_SURROGATES_SMOKE,
        surrogate_tail=SURROGATE_TAIL,
        random_state=SURROGATE_RANDOM_STATE,
        n_jobs_surrogates=N_JOBS_SURROGATES_SMOKE,
        surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
        store_surrogates=True,
        verbose=True,
    )
    if smoke_result is None:
        raise RuntimeError('Smoke compute did not return a result_dict')

    smoke_fig = generate_subject_plot_memory_outcome(
        subj_id=SMOKE_SUBJECT,
        roi_source=SMOKE_ROI_SOURCE,
        roi_target=SMOKE_ROI_TARGET,
        result_dict=smoke_result,
        epoch_data=smoke_result['epoch_data'],
        combined_df=combined_df,
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
        savepath=analysis_paths['base_dir'],
        event=ANALYSIS['event'],
    )

    smoke_pack = {
        'result': smoke_result,
        'smoke_df': smoke_df,
        'fig': smoke_fig,
        'cache_meta': smoke_meta,
    }
    SMOKE_RESULTS[analysis_tag] = smoke_pack
    ANALYSIS_RESULTS.setdefault(analysis_tag, {})['smoke_pack'] = smoke_pack
    smoke_df.to_csv(analysis_paths['smoke_df_csv'], index=False)
    with open(analysis_paths['smoke_meta_json'], 'w') as f:
        json.dump(smoke_meta, f, indent=2, default=str)

if RUN_SINGLE_SUBJECT_SMOKE and smoke_pack is not None:
    validate_smoke_outputs_memory_outcome(smoke_pack, combined_df)
    print('Smoke cache hit:', smoke_pack['cache_meta'].get('cache_hit'), '| file:', smoke_pack['cache_meta'].get('cache_file'))
    print('Smoke rows:', len(smoke_pack['smoke_df']))
    if 'gc_net_z' in smoke_pack['smoke_df'].columns:
        print('Smoke gc_net_z range:',
              float(np.nanmin(smoke_pack['smoke_df']['gc_net_z'])),
              float(np.nanmax(smoke_pack['smoke_df']['gc_net_z'])))
    display(smoke_pack['smoke_df'].head())
    plot_smoke_memory_outcome(smoke_pack)


In [ ]:
# Full batch run across ROI pairs / subjects
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'all_gc_df' in analysis_state:
    all_gc_df = analysis_state.get('all_gc_df', pd.DataFrame()).copy()
    cache_log_df = analysis_state.get('cache_log_df', pd.DataFrame()).copy()
    print('Loaded batch results from in-memory registry for', analysis_tag)
elif (not OVERWRITE_GC_DF_CACHE) and os.path.exists(analysis_paths['all_gc_csv']):
    all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
    print('Loaded batch results from analysis bundle:', analysis_paths['all_gc_csv'])
else:
    all_rows = []
    cache_log_rows = []

    for roi_source, roi_target in ROI_PAIRS:
        result_key = f'{roi_source}_{roi_target}_{analysis_tag}'
        n_with_results = 0

        for subj in subjects:
            _, _, meta = run_or_load_subject_gc_memory_outcome_df(
                subj_id=subj,
                roi_source=roi_source,
                roi_target=roi_target,
                base_dir=base_dir,
                combined_df=combined_df,
                cache_dir=GC_DF_CACHE_DIR,
                overwrite_cache=OVERWRITE_GC_DF_CACHE,
                return_result_dict=False,
                event=ANALYSIS['event'],
                day=ANALYSIS['day'],
                tmin=ANALYSIS['tmin'],
                tmax=ANALYSIS['tmax'],
                target_sfreq=ANALYSIS['target_sfreq'],
                fmin=ANALYSIS['fmin'],
                fmax=ANALYSIS['fmax'],
                gc_n_lags=ANALYSIS['gc_n_lags'],
                n_freqs=ANALYSIS['n_freqs'],
                min_trials=ANALYSIS['min_trials'],
                use_surrogates=USE_SURROGATES,
                n_surrogates=N_SURROGATES_BATCH,
                surrogate_tail=SURROGATE_TAIL,
                random_state=SURROGATE_RANDOM_STATE,
                n_jobs_surrogates=N_JOBS_SURROGATES_BATCH,
                surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
                store_surrogates=False,
                verbose=False,
            )

            cache_file = meta.get('cache_file', '')
            subj_df = load_gc_memory_outcome_df_cache(cache_file) if cache_file else None

            cache_log_rows.append({
                'participant': subj,
                'roi_source': roi_source,
                'roi_target': roi_target,
                'cache_hit': bool(meta.get('cache_hit', False)),
                'cache_file': cache_file,
                'loaded_from_cache': bool(subj_df is not None and len(subj_df) > 0),
            })

            if subj_df is None or len(subj_df) == 0:
                continue

            n_with_results += 1
            all_rows.append(subj_df)

        print(result_key, 'subjects with results:', n_with_results)

    all_gc_df = pd.concat(all_rows, ignore_index=True) if len(all_rows) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(cache_log_rows)
    all_gc_df.to_csv(analysis_paths['all_gc_csv'], index=False)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)
    print('Saved analysis bundle:', analysis_paths['all_gc_csv'])

analysis_state['all_gc_df'] = all_gc_df.copy()
analysis_state['cache_log_df'] = cache_log_df.copy()
print('Total exported rows:', len(all_gc_df))
if len(all_gc_df) > 0 and 'gc_net_z' in all_gc_df.columns:
    print('Batch gc_net_z finite rows:', int(all_gc_df['gc_net_z'].notna().sum()))
if len(cache_log_df) > 0:
    print('Cache hits:', int(cache_log_df['cache_hit'].sum()), '/', len(cache_log_df))
    print('Loaded rows from cache files:', int(cache_log_df['loaded_from_cache'].sum()), '/', len(cache_log_df))
    display(cache_log_df.head())


In [ ]:
import seaborn as sns
sns.lineplot(data=all_gc_df[(all_gc_df.roi_target=='ACC')], 
x='time', y='gc_net_z', hue='memory_outcome', errorbar='se')


In [ ]:
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'all_gc_df' in analysis_state:
    all_gc_df = analysis_state['all_gc_df'].copy()
else:
    all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
    analysis_state['all_gc_df'] = all_gc_df.copy()
print('Loaded all_gc_df rows:', len(all_gc_df), '| from', analysis_paths['all_gc_csv'])


In [ ]:
# Export hit/miss outputs and behavior-merged files
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if len(all_gc_df) == 0:
    print('No GC rows to export.')
    merged_gc_df = pd.DataFrame()
elif 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
    print('Loaded merged_gc_df from in-memory registry for', analysis_tag)
elif os.path.exists(analysis_paths['merged_gc_csv']):
    merged_gc_df = pd.read_csv(analysis_paths['merged_gc_csv'])
    print('Loaded merged_gc_df from analysis bundle:', analysis_paths['merged_gc_csv'])
else:
    all_gc_df.to_csv(analysis_paths['all_gc_csv'], index=False)
    print('Saved:', analysis_paths['all_gc_csv'])

    merged_rows = []
    for (roi_source, roi_target), pair_df in all_gc_df.groupby(['roi_source', 'roi_target']):
        merged = merge_with_behavior_memory_outcome(pair_df.copy(), combined_df)
        merged_rows.append(merged)
        pair_file = os.path.join(analysis_paths['base_dir'], f'{roi_source}_{roi_target}_mvgc_memory_outcome_merged.csv')
        merged.to_csv(pair_file, index=False)
    merged_gc_df = pd.concat(merged_rows, ignore_index=True) if len(merged_rows) > 0 else pd.DataFrame()
    merged_gc_df.to_csv(analysis_paths['merged_gc_csv'], index=False)
    print('Saved merged bundle:', analysis_paths['merged_gc_csv'])

analysis_state['all_gc_df'] = all_gc_df.copy()
analysis_state['merged_gc_df'] = merged_gc_df.copy()
if len(merged_gc_df) > 0:
    print('Merged rows:', len(merged_gc_df))
    display(merged_gc_df.head())


In [ ]:
# Aggregate plots + mixed-effects models (memory outcome)
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
if len(merged_gc_df) == 0:
    print('No merged data for aggregate/model plotting.')
else:
    source_roi = 'HPC'
    source_paths = get_analysis_paths(source_roi=source_roi)
    agg_df = merged_gc_df[merged_gc_df['roi_source'] == source_roi].copy()

    fig_agg = plot_aggregate_memory_outcome(
        agg_df,
        source_roi=source_roi,
        savepath=source_paths['base_dir'],
        event=ANALYSIS['event'],
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
    )
    if fig_agg is not None:
        plt.show()

    model_df = agg_df.copy()
    model_df['target_roi'] = model_df['roi_target']

    if ('gc_net_z' not in model_df.columns) or (not model_df['gc_net_z'].notna().any()):
        model_df['gc_net_z'] = model_df.groupby(['participant', 'roi_target'])['gc_net'].transform(
            lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
        )

    model_out = fit_mixed_effects_memory_outcome(model_df, reference='false_alarms')
    fig_mixed = plot_mixed_effects_memory_outcome(
        model_out,
        source_roi=source_roi,
        savepath=source_paths['base_dir'],
        event=ANALYSIS['event'],
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
    )
    if fig_mixed is not None:
        plt.show()

    analysis_state.setdefault('model_out', {})[source_roi] = model_out
    if len(model_out.get('stats_df', pd.DataFrame())) > 0:
        stats_file = source_paths['stats_csv']
        model_out['stats_df'].to_csv(stats_file, index=False)
        print('Saved mixed-model stats:', stats_file)
        display(model_out['stats_df'].head())
